 Generators are special functions that allow you to create iterators using the yield keyword instead of return. Rather than computing an entire dataset all at once and storing it in memory, generators compute values one at a time, on-demand (lazy evaluation), making them incredibly memory-efficient.

In [ ]:
# Generators  
squares = (x*x for x in range(1000000)) 

In [3]:
squares

<generator object <genexpr> at 0x111820e10>

In [32]:
next(squares)

36

In [25]:
# a generator that yields items instead of returning a list
def firstn(n):
    num = 0
    while num < n:
        yield num
        num += 1
   
sum_of_first_n = sum(firstn(1000000))

In [26]:
sum_of_first_n, type(sum_of_first_n)

(499999500000, int)

In [27]:
val = firstn(1000000)
for i in range(10):
    print(next(val))

0
1
2
3
4
5
6
7
8
9


In [28]:
def my_generator():
  yield 1
  yield 2
  yield 3

for value in my_generator():
  print(value)

1
2
3


In [33]:
def simple_gen():
  yield "Emil"
  yield "Tobias"
  yield "Linus"

gen = simple_gen()
print(next(gen))
print(next(gen))
print(next(gen))

Emil
Tobias
Linus


In [34]:
# Generator expression - creates a generator
gen_exp = (x * x for x in range(5))
print(gen_exp)
print(list(gen_exp))

<generator object <genexpr> at 0x111da8520>
[0, 1, 4, 9, 16]


In [35]:
# The send() method allows you to send a value to the generator:
def echo_generator():
  while True:
    received = yield
    print("Received:", received)

gen = echo_generator()
next(gen) # Prime the generator
gen.send("Hello")
gen.send("World")

Received: Hello
Received: World


In [36]:
# The close() method stops the generator:

def my_gen():
  try:
    yield 1
    yield 2
    yield 3
  finally:
    print("Generator closed")

gen = my_gen()
print(next(gen))
gen.close()

1
Generator closed


In [37]:
def simple_gen():
  yield 1
  yield 2

gen = simple_gen()
print(next(gen))
print(next(gen))
print(next(gen))  # This will raise StopIteration

1
2


StopIteration: 

Decorators let you add extra behavior to a function, without changing the function's code.

A decorator is a function that takes another function as input and returns a new function.



In [40]:
def changecase(func):
  def myinner():
    return func().upper()
  return myinner

@changecase
def myfunction():
  return "Hello Sally"

print(myfunction())

HELLO SALLY


In [41]:
@changecase
def otherfunction():
  return "I am speed!"

print(myfunction())
print(otherfunction())

HELLO SALLY
I AM SPEED!


Sometimes the decorator function has no control over the arguments passed from decorated function, to solve this problem, add (*args, **kwargs) to the wrapper function, this way the wrapper function can accept any number, and any type of arguments, and pass them to the decorated function.

In [42]:
def changecase(func):
  def myinner(*args, **kwargs):
    return func(*args, **kwargs).upper()
  return myinner

@changecase
def myfunction(nam):
  return "Hello " + nam

print(myfunction("John"))

HELLO JOHN


In [45]:
def changecase(n):
  def changecase(func):
    def myinner():
      if n == 1:
        a = func().lower()
      else:
        a = func().upper()
      return a
    return myinner
  return changecase

@changecase(1)
def myfunction():
  return "Hello Linus"

print(myfunction())

hello linus


In [49]:
# You can use multiple decorators on one function.
# This is done by placing the decorator calls on top of each other.
# Decorators are called in the reverse order, starting with the one closest to the function.

def changecase(func):
  def myinner():
    return func().upper()
  return myinner

def addgreeting(func):
  def myinner():
    return "Hello " + func() + " Have a good day!"
  return myinner

@changecase
@addgreeting
def myfunction():
  return "Tobias"

print(myfunction())


@addgreeting
@changecase
def myfunction2():
  return "Liam"

print(myfunction2())

HELLO TOBIAS HAVE A GOOD DAY!
Hello LIAM Have a good day!


In [52]:
# Functions in Python has metadata that can be accessed using the __name__ and __doc__ attributes.

def myfunction():
  return "Have a great day!"

print(myfunction.__name__)

myfunction


In [53]:
# But, when a function is decorated, the metadata of the original function is lost.

def changecase(func):
  def myinner():
    return func().upper()
  return myinner

@changecase
def myfunction():
  return "Have a great day!"

print(myfunction.__name__)

myinner


In [54]:
# To fix this, Python has a built-in function called functools.wraps 
# that can be used to preserve the original function's name and docstring

import functools

def changecase(func):
  @functools.wraps(func)
  def myinner():
    return func().upper()
  return myinner

@changecase
def myfunction():
  return "Have a great day!"

print(myfunction.__name__)

myfunction


In [78]:
import functools

def retry(n):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(n):
                print(attempt)
                try:
                    return func(*args, **kwargs)
                except Exception:
                    if attempt == n - 1:
                        raise                 # out of attempts: re-raise the last error
        return wrapper                        # return, don't call
    return decorator  

@retry(3)
def flaky():
    # raise ValueError
    print('just did')


flaky()

0
just did


@property lets you use a method as if it were a plain attribute: you write obj.age instead of obj.age(), but Python runs code behind the scenes.

In [55]:
class Circle:
    def __init__(self, r):
        self.r = r
    @property
    def area(self):            # now reads like a normal attribute
        return 3.14 * self.r ** 2

c = Circle(5)
print(c.area)      # no parentheses
c.r = 10
print(c.area)      # automatically recomputed: 314.0

78.5
314.0


Why use it

Computed values stay fresh. Change r and area updates itself, because it’s calculated each time.
Validation. A setter lets you block bad values:

In [56]:
class Person:
    def __init__(self, age):
        self.age = age          # goes through the setter below

    @property
    def age(self):
        return self._age

    @age.setter
    def age(self, value):
        if value < 0:
            raise ValueError("age can't be negative")
        self._age = value

p = Person(30)
p.age = -5      # raises ValueError

ValueError: age can't be negative

Read-only attributes: With a getter and no setter, p.area = 5 raises an error AttributeError.

Future-proofing: You can start with a plain attribute and later add logic without changing how callers use it.

Decorators in general do work on standalone functions; only @property needs a class.

In [66]:
import inspect
import functools

def log_call(func):
    @functools.wraps(func)
    def give_vals(*args, **kwargs):
        print(f"Calling {func.__name__} with args={args}, kwargs={kwargs}")
        return func(*args, **kwargs)
    return give_vals

@log_call
def get_sum(a,b):
    return a+b

print(get_sum(2,3))

Calling get_sum with args=(2, 3), kwargs={}
5


In [69]:
def add(x, lst=[]):      # BUG: one list shared across all calls
    lst.append(x); return lst

print(add(1)); 
print(add(2))           # [1, 2], not [2]

def add(x, lst=None):    # fix
    lst = [] if lst is None else lst
    lst.append(x); return lst

print(add(1)); 
print(add(2)) 
print(add(2, [4,5])) 

[1]
[1, 2]
[1]
[2]
[4, 5, 2]


In [73]:
# * unpacks a sequence at call time (f(*[1,2])), ** unpacks a dict.
# Regular parameters must come before *args:
def f(a, *args, **kwargs): 
    return a+sum(args), kwargs
f(1, 2, 3, k=4)   # a=1, args=(2,3), kwargs={'k': 4}

(6, {'k': 4})

In [74]:
def my_function(**myvar):
  print("Type:", type(myvar))
  print("Name:", myvar["name"])
  print("Age:", myvar["age"])
  print("All data:", myvar)

my_function(name = "Tobias", age = 30, city = "Bergen")

Type: <class 'dict'>
Name: Tobias
Age: 30
All data: {'name': 'Tobias', 'age': 30, 'city': 'Bergen'}


You can use both *args and **kwargs in the same function.

The order must be:

regular parameters, 
*args, 
**kwargs


In [75]:
def my_function(title, *args, **kwargs):
  print("Title:", title)
  print("Positional arguments:", args)
  print("Keyword arguments:", kwargs)

my_function("User Info", "Emil", "Tobias", age = 25, city = "Oslo")

Title: User Info
Positional arguments: ('Emil', 'Tobias')
Keyword arguments: {'age': 25, 'city': 'Oslo'}


Using * to unpack a list into arguments:

Using ** to unpack a dictionary into keyword arguments:




class Timer:
    
    def __enter__(self): 
        self.t = time.perf_counter(); return self

    def __exit__(self, exc_type, exc, tb):
        print(time.perf_counter() - self.t)
        return False          # False = don't swallow exceptions


with Timer(): ...

with guarantees __exit__ runs even on error. Example: file handles, GPU/no_grad blocks (torch.no_grad() is one).

“Dunder” is short for double underscore. Dunder methods are methods with names like __init__ or __len__, wrapped in double underscores. Also called “magic methods”.

The idea: you don’t call them directly. Python calls them for you when you use normal syntax. They let your own classes behave like built-in types.

In [76]:
class Playlist:
    def __init__(self, songs):
        self.songs = songs
    def __len__(self):
        return len(self.songs)
    def __getitem__(self, i):
        return self.songs[i]
    def __repr__(self):
        return f"Playlist({self.songs})"

p = Playlist(["a", "b", "c"])
len(p)     # 3, runs __len__
p[1]       # 'b', runs __getitem__
p          # Playlist(['a', 'b', 'c']), runs __repr__

Playlist(['a', 'b', 'c'])

Without those methods, len(p) and p[1] would fail.

Why it matters for your interview:

PyTorch Dataset needs exactly two: 
__len__ (how many samples) and __getitem__ (get sample i). DataLoader calls them for you.
nn.Module uses __call__, which is why model(x) works and runs forward.

How it works:

When you write len(x), Python’s rule is “look for a method named __len__ on type(x) and call it”.

When you write x[3], the rule is “call __getitem__”.

When you write x(...), the rule is “call __call__”.